# Image Classification Pipeline

## 1. Project Overview
This notebook implements an image classification pipeline using TensorFlow/Keras. We will:
1.  Download and extract the dataset.
2.  Preprocess and augment the data.
3.  Train three different CNN architectures:
    * **Model 1:** A simple custom CNN (Baseline).
    * **Model 2:** Transfer Learning with ResNet50.
    * **Model 3:** Transfer Learning with EfficientNetB0.
4.  Evaluate each model using Accuracy, Precision, Recall, and F1-Score.
5.  Compile the results and generate a PDF report.

In [1]:
# Install necessary libraries
!pip install -q fpdf

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, precision_recall_fscore_support
import numpy as np
import matplotlib.pyplot as plt
import gdown
import zipfile
import os
import pandas as pd
from fpdf import FPDF

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

## 2. Data Loading
We download the dataset directly from Google Drive using the file ID.

In [3]:
# Download the dataset
file_id = '1CjhNulNquoA8gWQtmHD83ib5t8lGBGC1'
output_file = 'animal_data.zip'

if not os.path.exists(output_file):
    gdown.download(id=file_id, output=output_file, quiet=False)

# Unzip
if not os.path.exists('animal_data'):
    with zipfile.ZipFile(output_file, 'r') as zip_ref:
        zip_ref.extractall()
    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

Dataset already extracted.


In [4]:
# Define parameters
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = 'animal_data'  # Check if this is the correct root after unzipping

# Load datasets
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' # Use categorical for metrics calculation ease
)

val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print(f"Classes found: {class_names}")

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 1944 files belonging to 2 classes.
Using 1556 files for training.
Found 1944 files belonging to 2 classes.
Using 388 files for validation.
Classes found: ['__MACOSX', 'animal_data']


## 3. Evaluation Helper Function
This function will help us calculate the required metrics for any model.

In [5]:
def evaluate_model(model, dataset, model_name="Model"):
    print(f"Evaluating {model_name}...")
    y_true = []
    y_pred = []
    
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        
    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    accuracy = np.mean(np.array(y_true) == np.array(y_pred))
    
    print(f"{model_name} Results:")
    print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

## 4. Model Architectures

### Model 1: Custom CNN (Baseline)

In [6]:
def build_custom_cnn(num_classes):
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_1 = build_custom_cnn(len(class_names))
history_1 = model_1.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

d:\Users\Nishant\anaconda3\Lib\site-packages\keras\src\layers\preprocessing\tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 32s 555ms/step - accuracy: 0.9910 - loss: 0.0141 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 29s 602ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 32s 660ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 30s 602ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 30s 607ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 31s 637ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 30s 601ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 29s 583

### Model 2: ResNet50 (Transfer Learning)

In [7]:
def build_resnet_model(num_classes):
    base_model = applications.ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False  # Freeze base layers
    
    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = applications.resnet50.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_2 = build_resnet_model(len(class_names))
history_2 = model_2.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.9865 - loss: 0.0254 - val_accuracy: 1.0000 - val_loss: 1.1187e-05
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 1.0000 - loss: 7.8865e-06 - val_accuracy: 1.0000 - val_loss: 1.0265e-05
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 1.0000 - loss: 7.7037e-06 - val_accuracy: 1.0000 - val_loss: 1.0191e-05
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 66s 1s/step - accuracy: 1.0000 - loss: 7.9975e-06 - val_accuracy: 1.0000 - val_loss: 1.0107e-05
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 1.0000 - loss: 9.4392e-06 - val_accuracy: 1.0000 - val_loss: 9.9924e-06
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 1.0000 - loss: 7.4647e-06 - val_accuracy: 1.0000 - val_loss: 9.8831e-06
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 1.0000 - loss: 7.2787e-06 - val_accuracy: 1.0000 - val_loss: 9.7864e-06
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 1.00

### Model 3: EfficientNetB0 (Transfer Learning)

In [13]:
def build_efficientnet_model(num_classes):
    # Avoid loading imagenet weights here to prevent channel-shape mismatches
    base_model = applications.EfficientNetB0(weights=None, include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    # Scale inputs to [0,1] before feeding the base model
    x = layers.Rescaling(1./255)(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_3 = build_efficientnet_model(len(class_names))
history_3 = model_3.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

Epoch 1/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 45s 652ms/step - accuracy: 0.9801 - loss: 0.6696 - val_accuracy: 1.0000 - val_loss: 0.6456
Epoch 2/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 22s 441ms/step - accuracy: 1.0000 - loss: 0.6236 - val_accuracy: 1.0000 - val_loss: 0.6014
Epoch 3/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 469ms/step - accuracy: 1.0000 - loss: 0.5812 - val_accuracy: 1.0000 - val_loss: 0.5605
Epoch 4/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 24s 488ms/step - accuracy: 1.0000 - loss: 0.5418 - val_accuracy: 1.0000 - val_loss: 0.5228
Epoch 5/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 471ms/step - accuracy: 1.0000 - loss: 0.5055 - val_accuracy: 1.0000 - val_loss: 0.4879
Epoch 6/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 472ms/step - accuracy: 1.0000 - loss: 0.4718 - val_accuracy: 1.0000 - val_loss: 0.4557
Epoch 7/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 481ms/step - accuracy: 1.0000 - loss: 0.4409 - val_accuracy: 1.0000 - val_loss: 0.4260
Epoch 8/10
49/49 ━━━━━━━━━━━━━━━━━━━━ 23s 470ms/step - accuracy: 1.0000 - loss: 0.4124 - val_accu

## 5. Compilation of Results
We evaluate all models and compile the metrics.

In [17]:
results = []
results.append(evaluate_model(model_1, val_ds, "Custom CNN"))
results.append(evaluate_model(model_2, val_ds, "ResNet50"))
results.append(evaluate_model(model_3, val_ds, "EfficientNetB0"))

df_results = pd.DataFrame(results)
print("\nCombined Results:")
display(df_results)

Evaluating Custom CNN...
Custom CNN Results:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000
Evaluating ResNet50...
ResNet50 Results:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000
Evaluating EfficientNetB0...
EfficientNetB0 Results:
Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1: 1.0000

Combined Results:


,Model,Accuracy,Precision,Recall,F1 Score
0,Custom CNN,1.0,1.0,1.0,1.0
1,ResNet50,1.0,1.0,1.0,1.0
2,EfficientNetB0,1.0,1.0,1.0,1.0


## 6. Generate PDF Report
This section generates a PDF file `results_report.pdf` summarizing the findings.

In [18]:
class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, 'Image Classification Model Report', 0, 1, 'C')

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

pdf = PDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

# Title Section
pdf.set_font("Arial", 'B', 16)
pdf.cell(200, 10, txt="Model Evaluation Summary", ln=True, align='C')
pdf.ln(10)

# Metrics Table
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, txt="Performance Metrics:", ln=True, align='L')
pdf.ln(5)

# Header
pdf.set_font("Arial", 'B', 10)
pdf.cell(40, 10, "Model", 1)
pdf.cell(30, 10, "Accuracy", 1)
pdf.cell(30, 10, "Precision", 1)
pdf.cell(30, 10, "Recall", 1)
pdf.cell(30, 10, "F1 Score", 1)
pdf.ln()

# Rows
pdf.set_font("Arial", size=10)
for index, row in df_results.iterrows():
    pdf.cell(40, 10, str(row['Model']), 1)
    pdf.cell(30, 10, f"{row['Accuracy']:.4f}", 1)
    pdf.cell(30, 10, f"{row['Precision']:.4f}", 1)
    pdf.cell(30, 10, f"{row['Recall']:.4f}", 1)
    pdf.cell(30, 10, f"{row['F1 Score']:.4f}", 1)
    pdf.ln()

pdf.ln(10)
pdf.set_font("Arial", size=12)
best_model = df_results.loc[df_results['F1 Score'].idxmax()]['Model']
pdf.cell(0, 10, f"Conclusion: The best performing model is {best_model}.", 0, 1)

pdf.output("results_report2.pdf")
print("PDF Report 'results_report2.pdf' generated successfully.")

PDF Report 'results_report2.pdf' generated successfully.
